# GPU SHA-256 Project — Day 1 Setup Notebook

**Goal of this notebook:** confirm everyone's Colab can compile and run CUDA on a real GPU.
Every team member should run this top-to-bottom on Day 1. If the last cell prints
`SETUP OK`, your toolchain works and you can start your task.

### First: turn on the GPU
Menu: **Runtime → Change runtime type → Hardware accelerator → GPU → Save.**
Then run the cells below in order (Shift+Enter).

## Step 1 — Check which GPU we got
On free Colab this is usually a **Tesla T4** (16 GB). That is plenty for this project.

In [ ]:
!nvidia-smi

## Step 2 — Check the CUDA compiler (nvcc) is installed
`nvcc` is the compiler that turns our `.cu` files into GPU programs. Colab has it preinstalled.

In [ ]:
!nvcc --version

## Step 3 — Write a tiny CUDA program
This demonstrates the EXACT pattern your SHA-256 kernel will use:
**one thread per item**. Here each thread adds 1 to its own number.

The `%%writefile` magic saves the cell's contents to a file called `hello.cu`.

In [ ]:
%%writefile hello.cu
#include <cstdio>

// ---- THE KERNEL: runs on the GPU, once per thread ----
// __global__ marks this as a kernel the CPU can launch.
__global__ void addOne(int* data, int n) {
    // Each thread computes its own unique ID. This is the standard CUDA idiom.
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    // Guard: some threads have i >= n and must do nothing.
    // (Same guard your SHA-256 kernel uses for out-of-range threads.)
    if (i < n) {
        data[i] = data[i] + 1;   // do MY work on MY element
    }
}

int main() {
    const int N = 10;
    int h_data[N];                       // host (CPU) array
    for (int i = 0; i < N; i++) h_data[i] = i;   // 0,1,2,...,9

    // 1. Allocate memory ON THE GPU
    int* d_data;
    cudaMalloc(&d_data, N * sizeof(int));

    // 2. Copy data CPU -> GPU
    cudaMemcpy(d_data, h_data, N * sizeof(int), cudaMemcpyHostToDevice);

    // 3. Launch the kernel: enough threads to cover all N elements
    int threadsPerBlock = 256;
    int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    addOne<<<numBlocks, threadsPerBlock>>>(d_data, N);

    // 4. Copy results GPU -> CPU
    cudaMemcpy(h_data, d_data, N * sizeof(int), cudaMemcpyDeviceToHost);
    cudaFree(d_data);

    // 5. Print: should be 1,2,3,...,10
    printf("Result after GPU addOne: ");
    for (int i = 0; i < N; i++) printf("%d ", h_data[i]);
    printf("\n");
    return 0;
}


## Step 4 — Compile it with nvcc

In [ ]:
!nvcc -arch=sm_75 hello.cu -o hello

> Note: `-arch=sm_75` targets the Tesla T4. If Colab gives you a different GPU and
> compilation complains, just drop the `-arch` flag: `!nvcc hello.cu -o hello`

## Step 5 — Run it
Expected output: `Result after GPU addOne: 1 2 3 4 5 6 7 8 9 10`

In [ ]:
!./hello

## Step 6 — Confirm the CPU reference tool works (for M2 / M4)
Python's `hashlib` is our trusted SHA-256. This is what every GPU result is checked against.
These two lines must match the test vectors in the I/O contract.

In [ ]:
import hashlib

print('sha256("")    =', hashlib.sha256(b'').hexdigest())
print('sha256("abc") =', hashlib.sha256(b'abc').hexdigest())

assert hashlib.sha256(b'').hexdigest()    == 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'
assert hashlib.sha256(b'abc').hexdigest() == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'

print('\nSETUP OK — GPU compiles + runs, and the CPU reference matches the test vectors.')